# Challenge 5 — Clean & export the flight + weather dataset

**Goal:** turn `Flight_Data_With_Weather_Final.csv` into clean, documented tables that the AAR model, the delay-risk classifier and the dashboard can all build on, without leaking future information.

What this notebook does, in order:

1. Profiles the raw file and shows what it actually covers.
2. Keeps the useful columns and gives every dropped column a written reason.
3. Builds proper UTC timestamps for every flight, using each airport's time zone.
4. **Fixes how the weather is labelled.** The provided weather is indexed in UTC but was joined on *local* clock time, so each snapshot is really 4–30 h older than the flight. We record its true observation time and age.
5. Replaces sentinel zeros (gust, snow, condition code) with missing values.
6. Pulls hourly METARs for the hubs from Iowa Environmental Mesonet (IEM). This adds wind direction, visibility and ceiling, which the provided file lacks, so we can compute crosswind and tailwind for each runway.
7. Builds targets (`> 45 min` severe delay, cancellations, diversions).
8. Exports Parquet/CSV files, a data dictionary and a cleaning report to `data/processed/`.

Columns from BTS keep their original UPPERCASE names so they trace back to the BTS documentation. Every column we derive is lowercase.

## 1. Setup

In [1]:
from pathlib import Path
import json
import re
import urllib.parse
import urllib.request

import numpy as np
import pandas as pd
import airportsdata

RAW_CSV = Path("Flight_Data_With_Weather_Final.csv")
OUT_DIR = Path("data/processed")
EXT_DIR = Path("data/external")   # cache for downloaded public data
OUT_DIR.mkdir(parents=True, exist_ok=True)
EXT_DIR.mkdir(parents=True, exist_ok=True)

# Airports in scope. Runway headings follow the brief's simplifying assumption.
# To add an airport, add its IATA code and a heading; nothing else in the pipeline is hub-specific.
HUBS = {
    "ATL": {"runway_heading_deg": 80},
    "DFW": {"runway_heading_deg": 180},
}
SEVERE_DELAY_MIN = 45
DOWNLOAD_METARS = True   # set False to run fully offline (skips section 7)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)

## 2. Load and profile the raw file

In [2]:
raw = pd.read_csv(RAW_CSV, low_memory=False).copy()   # copy() defragments the frame
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")
print("Dates:", raw.FL_DATE.min(), "to", raw.FL_DATE.max(), f"({raw.FL_DATE.nunique()} days)")

is_hub_dep = raw.ORIGIN.isin(HUBS)
is_hub_arr = raw.DEST.isin(HUBS)
coverage = pd.Series({
    "all rows": len(raw),
    "departs a hub": is_hub_dep.sum(),
    "arrives at a hub": is_hub_arr.sum(),
    "touches a hub (either)": (is_hub_dep | is_hub_arr).sum(),
    "hub to hub": (is_hub_dep & is_hub_arr).sum(),
})
display(coverage.to_frame("rows"))

n_days = raw.FL_DATE.nunique()
per_day = pd.DataFrame({
    "departures/day in file": raw[is_hub_dep].groupby("ORIGIN").size() / n_days,
    "arrivals/day in file": raw[is_hub_arr].groupby("DEST").size() / n_days,
}).round(1)
display(per_day)

148,273 rows x 123 columns
Dates: 2023-01-01 to 2025-07-31 (943 days)


,rows
all rows,148273
departs a hub,16151
arrives at a hub,9461
touches a hub (either),25299
hub to hub,313


,departures/day in file,arrivals/day in file
ATL,8.9,5.8
DFW,8.2,4.3


**What this shows:**

* Only about 17% of rows involve ATL or DFW. The rest are a sample of the whole US network (136 origins, 350 destinations).
* The file is a **sample of roughly 1% of flights**. ATL really handles about 1,000 departures a day, but the file has about 9. That is enough to train a flight-level delay classifier. It is **not enough to measure hourly arrival rates (AAR)**, which needs the full BTS data (see `PLAN.md`).
* We keep all rows in one export, so we can test whether the pipeline generalises beyond the two hubs, and also export a hub-only file.

## 3. Column audit: what to keep and why

In [3]:
audit = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "null_frac": raw.isna().mean().round(3),
    "n_unique": raw.nunique(),
})
display(audit)

,dtype,null_frac,n_unique
YEAR,int64,0.000,3
QUARTER,int64,0.000,4
MONTH,int64,0.000,12
DAY_OF_MONTH,int64,0.000,31
DAY_OF_WEEK,int64,0.000,7
FL_DATE,str,0.000,943
OP_UNIQUE_CARRIER,str,0.000,15
OP_CARRIER_AIRLINE_ID,int64,0.000,15
OP_CARRIER,str,0.000,15
TAIL_NUM,str,0.002,6308


In [4]:
# Redundancy and sentinel checks that justify the drops below
checks = {
    "OP_CARRIER == OP_UNIQUE_CARRIER": (raw.OP_CARRIER == raw.OP_UNIQUE_CARRIER).all(),
    "FLIGHTS is always 1": (raw.FLIGHTS == 1).all(),
    "snow_ORIGIN and snow_DEST are always 0": ((raw.snow_ORIGIN == 0) & (raw.snow_DEST == 0)).all(),
    "DEP_DELAY_NEW == max(DEP_DELAY, 0)": np.allclose(raw.DEP_DELAY_NEW.dropna(), raw.DEP_DELAY.dropna().clip(lower=0)),
    "DEP_DEL15 == (DEP_DELAY >= 15)": ((raw.DEP_DELAY >= 15).astype(float) == raw.DEP_DEL15)[raw.DEP_DELAY.notna()].all(),
    "delay causes only filled when ARR_DEL15 == 1": (raw.CARRIER_DELAY.notna() == (raw.ARR_DEL15 == 1)).all(),
    "DIV3..DIV5 columns entirely empty": raw.filter(regex=r"^DIV[345]_").isna().all().all(),
}
display(pd.Series(checks, name="holds"))

OP_CARRIER == OP_UNIQUE_CARRIER                 True
FLIGHTS is always 1                             True
snow_ORIGIN and snow_DEST are always 0          True
DEP_DELAY_NEW == max(DEP_DELAY, 0)              True
DEP_DEL15 == (DEP_DELAY >= 15)                  True
delay causes only filled when ARR_DEL15 == 1    True
DIV3..DIV5 columns entirely empty               True
Name: holds, dtype: bool

In [5]:
# Every kept column gets a role. The role tells the modelling step what it may use as an input:
#   id         : keys and identifiers
#   schedule   : known before the day of operation, so safe to use as a feature
#   outcome    : only known after the flight happened. Use as a label or for analysis, NEVER as a
#                feature for the same flight (lagged aggregates from earlier flights are fine)
#   weather    : provided weather snapshot (see section 5 for its true timing)
KEEP = {
    "FL_DATE":            ("id",       "Scheduled departure date, local time at origin"),
    "OP_UNIQUE_CARRIER":  ("schedule", "Marketing/operating carrier code"),
    "OP_CARRIER_FL_NUM":  ("id",       "Flight number"),
    "TAIL_NUM":           ("schedule", "Aircraft registration; links consecutive legs for delay propagation"),
    "ORIGIN":             ("schedule", "Origin airport (IATA)"),
    "DEST":               ("schedule", "Destination airport (IATA)"),
    "CRS_DEP_TIME":       ("schedule", "Scheduled departure, local hhmm"),
    "CRS_ARR_TIME":       ("schedule", "Scheduled arrival, local hhmm at destination"),
    "CRS_ELAPSED_TIME":   ("schedule", "Scheduled block time, minutes"),
    "DISTANCE":           ("schedule", "Great-circle distance, miles"),
    "DEP_TIME":           ("outcome",  "Actual departure, local hhmm"),
    "DEP_DELAY":          ("outcome",  "Departure delay, minutes (negative = early)"),
    "TAXI_OUT":           ("outcome",  "Taxi-out minutes; a ground-congestion signal when lagged"),
    "WHEELS_OFF":         ("outcome",  "Wheels-off, local hhmm"),
    "WHEELS_ON":          ("outcome",  "Wheels-on, local hhmm at destination"),
    "TAXI_IN":            ("outcome",  "Taxi-in minutes"),
    "ARR_TIME":           ("outcome",  "Actual arrival, local hhmm"),
    "ARR_DELAY":          ("outcome",  "Arrival delay, minutes"),
    "ACTUAL_ELAPSED_TIME":("outcome",  "Actual block time, minutes"),
    "AIR_TIME":           ("outcome",  "Airborne minutes; includes holding"),
    "CANCELLED":          ("outcome",  "1 = cancelled"),
    "CANCELLATION_CODE":  ("outcome",  "A carrier, B weather, C NAS, D security"),
    "DIVERTED":           ("outcome",  "1 = diverted"),
    "CARRIER_DELAY":      ("outcome",  "Cause minutes (BTS fills only when ARR_DELAY >= 15)"),
    "WEATHER_DELAY":      ("outcome",  "Cause minutes, extreme weather"),
    "NAS_DELAY":          ("outcome",  "Cause minutes, National Airspace System (includes non-extreme weather/ATC programs)"),
    "SECURITY_DELAY":     ("outcome",  "Cause minutes"),
    "LATE_AIRCRAFT_DELAY":("outcome",  "Cause minutes, late inbound aircraft"),
}
WX_RAW = ["temp", "prcp", "wspd", "wpgt", "pres", "coco"]
for side in ["ORIGIN", "DEST"]:
    for v in WX_RAW:
        KEEP[f"{v}_{side}"] = ("weather", "renamed below")

# Every dropped column gets a reason, so nothing disappears silently.
DROP_RULES = [
    (r"^DIV\d_|^DIV_(REACHED_DEST|ACTUAL_ELAPSED_TIME|ARR_DELAY|DISTANCE)$|^DIV_AIRPORT_LANDINGS$",
        ">99.7% empty diversion detail; DIVERTED flag is kept"),
    (r"^(FIRST_DEP_TIME|TOTAL_ADD_GTIME|LONGEST_ADD_GTIME)$", "99.3% empty gate-return detail"),
    (r"_(AIRPORT_ID|AIRPORT_SEQ_ID|CITY_MARKET_ID|STATE_FIPS|STATE_NM|STATE_ABR|WAC|CITY_NAME)$",
        "Redundant with the IATA code; airport metadata is re-joined from airportsdata"),
    (r"^(OP_CARRIER|OP_CARRIER_AIRLINE_ID)$", "Duplicates OP_UNIQUE_CARRIER"),
    (r"^(YEAR|QUARTER|MONTH|DAY_OF_MONTH|DAY_OF_WEEK)$", "Re-derived from proper timestamps"),
    (r"^(DEP|ARR)_(DELAY_NEW|DEL15|DELAY_GROUP|TIME_BLK)$|^DISTANCE_GROUP$", "Simple transforms of DEP_DELAY / ARR_DELAY / CRS times / DISTANCE"),
    (r"^FLIGHTS$", "Constant (always 1)"),
    (r"^snow_", "Constant 0 in every row: missing data filled with 0, not real snow depth"),
]
dropped = {}
for col in raw.columns:
    if col in KEEP:
        continue
    reason = next((r for pat, r in DROP_RULES if re.search(pat, col)), None)
    assert reason is not None, f"Column {col} has no keep/drop decision"
    dropped[col] = reason

drop_table = pd.Series(dropped, name="reason").rename_axis("column").reset_index()
display(drop_table.groupby("reason").column.apply(lambda s: ", ".join(s) if len(s) <= 6 else f"{len(s)} cols: {', '.join(s[:4])}, ...").to_frame())
print(f"Keeping {len(KEEP)} of {raw.shape[1]} columns, dropping {len(dropped)}")

df = raw[list(KEEP)].copy()

,column
reason,
99.3% empty gate-return detail,"FIRST_DEP_TIME, TOTAL_ADD_GTIME, LONGEST_ADD_G..."
>99.7% empty diversion detail; DIVERTED flag is kept,"45 cols: DIV_AIRPORT_LANDINGS, DIV_REACHED_DES..."
Constant (always 1),FLIGHTS
"Constant 0 in every row: missing data filled with 0, not real snow depth","snow_ORIGIN, snow_DEST"
Duplicates OP_UNIQUE_CARRIER,"OP_CARRIER_AIRLINE_ID, OP_CARRIER"
Re-derived from proper timestamps,"YEAR, QUARTER, MONTH, DAY_OF_MONTH, DAY_OF_WEEK"
Redundant with the IATA code; airport metadata is re-joined from airportsdata,"16 cols: ORIGIN_AIRPORT_ID, ORIGIN_AIRPORT_SEQ..."
Simple transforms of DEP_DELAY / ARR_DELAY / CRS times / DISTANCE,"9 cols: DEP_DELAY_NEW, DEP_DEL15, DEP_DELAY_GR..."


Keeping 40 of 123 columns, dropping 83


## 4. Weather columns: rename, units, sentinel values

The weather is **Meteostat hourly** data. Units: temperature in °C, precipitation in mm, wind and gust in **km/h**, pressure in hPa. `coco` is Meteostat's weather condition code (1–27).

Problems found:
* `wpgt` (gust) is 0 in 98% of rows. Gusts appear 0% of the time in 2023, 0.1% in 2024 and 9% in 2025, so the zeros mean "not reported", not calm air → set to missing.
* `coco = 0` is not a valid code → missing.
* `snow` is always 0 → already dropped.
* **There is no wind direction column**, so crosswind and tailwind cannot be computed from this file alone. Section 7 fixes this with METARs.

In [6]:
gust_by_year = raw.groupby("YEAR").apply(lambda g: (g.wpgt_ORIGIN > 0).mean(), include_groups=False)
display(gust_by_year.rename("share of rows with gust > 0").to_frame())

WX_NAMES = {"temp": "temp_c", "prcp": "precip_mm", "wspd": "wind_kmh", "wpgt": "gust_kmh",
            "pres": "pres_hpa", "coco": "cond_code"}
COCO_LABEL = {1: "clear", 2: "fair", 3: "cloudy", 4: "overcast", 5: "fog", 6: "freezing fog",
              7: "light rain", 8: "rain", 9: "heavy rain", 10: "freezing rain", 11: "heavy freezing rain",
              12: "sleet", 13: "heavy sleet", 14: "light snowfall", 15: "snowfall", 16: "heavy snowfall",
              17: "rain shower", 18: "heavy rain shower", 19: "sleet shower", 20: "heavy sleet shower",
              21: "snow shower", 22: "heavy snow shower", 23: "lightning", 24: "hail",
              25: "thunderstorm", 26: "heavy thunderstorm", 27: "storm"}
COCO_GROUP = {**{c: "none" for c in (1, 2, 3, 4)}, 5: "low_visibility",
              **{c: "rain" for c in (7, 8, 9, 17, 18)},
              **{c: "winter" for c in (6, 10, 11, 12, 13, 14, 15, 16, 19, 20, 21, 22)},
              **{c: "convective" for c in (23, 24, 25, 26, 27)}}

for side in ["ORIGIN", "DEST"]:
    p = side.lower()
    df = df.rename(columns={f"{k}_{side}": f"{p}_wx_{v}" for k, v in WX_NAMES.items()})
    df[f"{p}_wx_gust_kmh"] = df[f"{p}_wx_gust_kmh"].where(df[f"{p}_wx_gust_kmh"] > 0)
    df[f"{p}_wx_cond_code"] = df[f"{p}_wx_cond_code"].where(df[f"{p}_wx_cond_code"].between(1, 27)).astype("Int8")
    df[f"{p}_wx_wind_kt"] = (df[f"{p}_wx_wind_kmh"] / 1.852).round(1)   # ATC works in knots
    df[f"{p}_wx_cond_group"] = df[f"{p}_wx_cond_code"].map(COCO_GROUP).astype("category")

display(df.origin_wx_cond_group.value_counts(dropna=False).to_frame("origin rows"))

,share of rows with gust > 0
YEAR,
2023,0.000000
2024,0.000961
2025,0.092744


,origin rows
origin_wx_cond_group,
none,126845
rain,12794
low_visibility,5641
winter,1368
NaN,1145
convective,480


## 5. Timestamps: local → UTC

BTS times are **local clock times at each airport**, given as `hhmm` integers. To compare across time zones, and to join with weather or forecasts, everything needs a UTC timestamp.

* Time zones come from the `airportsdata` package (offline, covers IATA and ICAO). PBI is added by hand.
* Scheduled arrival = scheduled departure + scheduled block time. This handles overnight and cross-time-zone flights without any date guessing.
* Actual times = scheduled + delay minutes. This avoids the `2400` / next-day parsing traps in `DEP_TIME` and `ARR_TIME`.

In [7]:
AIRPORTS = airportsdata.load("IATA")
OVERRIDES = {"PBI": {"tz": "America/New_York", "icao": "KPBI"}}

codes = pd.unique(df[["ORIGIN", "DEST"]].values.ravel())
meta = {c: OVERRIDES.get(c) or AIRPORTS.get(c) for c in codes}
missing = [c for c, m in meta.items() if not m]
assert not missing, f"No time zone for {missing}"
TZ = {c: m["tz"] for c, m in meta.items()}
ICAO = {c: m["icao"] for c, m in meta.items()}

df["origin_icao"] = df.ORIGIN.map(ICAO)
df["dest_icao"] = df.DEST.map(ICAO)
df["origin_tz"] = df.ORIGIN.map(TZ)
df["dest_tz"] = df.DEST.map(TZ)


def hhmm_to_naive(dates, hhmm):
    hhmm = hhmm.astype(int)
    return (pd.to_datetime(dates) + pd.to_timedelta(hhmm // 100, unit="h")
            + pd.to_timedelta(hhmm % 100, unit="m"))


def local_to_utc(naive, tz_series):
    out = pd.Series(pd.NaT, index=naive.index, dtype="datetime64[ns, UTC]")
    for tz, idx in tz_series.groupby(tz_series).groups.items():
        s = naive.loc[idx]
        # DST edge cases: pick standard time for ambiguous hours, shift forward over the spring gap
        out.loc[idx] = (s.dt.tz_localize(tz, ambiguous=np.zeros(len(s), dtype=bool),
                                         nonexistent="shift_forward").dt.tz_convert("UTC"))
    return out


df["sched_dep_utc"] = local_to_utc(hhmm_to_naive(df.FL_DATE, df.CRS_DEP_TIME), df.origin_tz)
df["sched_arr_utc"] = df.sched_dep_utc + pd.to_timedelta(df.CRS_ELAPSED_TIME, unit="m")
df["actual_dep_utc"] = df.sched_dep_utc + pd.to_timedelta(df.DEP_DELAY, unit="m")
df["actual_arr_utc"] = df.sched_arr_utc + pd.to_timedelta(df.ARR_DELAY, unit="m")


def to_local(ts_utc, tz_series):
    out = pd.Series(index=ts_utc.index, dtype="object")
    for tz, idx in tz_series.groupby(tz_series).groups.items():
        out.loc[idx] = ts_utc.loc[idx].dt.tz_convert(tz).dt.tz_localize(None)
    return pd.to_datetime(out)


df["sched_dep_local"] = to_local(df.sched_dep_utc, df.origin_tz)
df["sched_arr_local"] = to_local(df.sched_arr_utc, df.dest_tz)

# Sanity check: rebuilt local arrival time should match the BTS CRS_ARR_TIME
rebuilt = df.sched_arr_local.dt.hour * 100 + df.sched_arr_local.dt.minute
match = (rebuilt == df.CRS_ARR_TIME % 2400).mean()
print(f"Rebuilt scheduled arrival matches CRS_ARR_TIME for {match:.2%} of flights")
assert match > 0.99

# Calendar features from local schedule (known in advance)
df["sched_dep_hour_local"] = df.sched_dep_local.dt.hour.astype("int8")
df["sched_arr_hour_local"] = df.sched_arr_local.dt.hour.astype("int8")
df["month"] = df.sched_dep_local.dt.month.astype("int8")
df["day_of_week"] = df.sched_dep_local.dt.day_of_week.astype("int8")   # 0 = Monday

Rebuilt scheduled arrival matches CRS_ARR_TIME for 100.00% of flights


## 6. When was the provided weather observed?

**The brief warns against using weather recorded when the flight arrives.** So we checked when each weather value was actually observed.

**Finding:** the weather was joined on `(airport, FL_DATE, scheduled local hour)`, but Meteostat indexes its rows in **UTC**. The "08:00" weather on an 08:00-local Atlanta departure is really the 08:00 UTC observation, which is 03:00–04:00 local.

Evidence:
1. **Daily temperature cycle.** At LAX the provided temperature is lowest at "13:00" and highest at "21:00". Those are 05:00/06:00 and 13:00/14:00 Pacific time, which is exactly where real LAX temperatures bottom out and peak.
2. **Join key test.** Destination weather matches the destination's origin-side weather at the *scheduled* arrival hour 99.9% of the time, but only about 70% of the time at the actual arrival hour. Overnight arrivals reuse the *departure* date.
3. **Real METARs.** Section 7 compares the values with IEM METARs for ATL/DFW under both readings: the UTC reading matches, the local one does not.

**What it means:**
* Good news: the attached weather is **always from before the flight**, so it is not future information.
* Bad news: it is **not "weather at departure"**. Its age ranges from about 4 h (ATL in summer) to over 30 h (overnight arrivals), and it varies by time zone, daylight saving time and schedule. Feeding it to a model as "current weather" would teach the wrong relationship.
* The fix: record the true observation time (`*_wx_obs_utc`) and its age (`*_wx_age_h`). A feature is valid for forecast horizon `k` only if `age >= k`. For proper modelling, use the hourly METAR/TAF series and an as-of join at the forecast issue time (see the preview in section 9).

In [8]:
# Evidence 1: diurnal cycle by the provided "local" hour label
hour_label = raw.CRS_DEP_TIME // 100
diurnal = (raw.assign(h=hour_label)[raw.ORIGIN.isin(["LAX", "ATL", "DFW"])]
           .groupby(["ORIGIN", "h"]).temp_ORIGIN.mean().unstack(0).round(1))
print("Hour label of min / max mean temperature:")
display(pd.DataFrame({"coldest label hour": diurnal.idxmin(), "warmest label hour": diurnal.idxmax()}))

# Evidence 2: which key reproduces the destination weather?
o_key = (raw.assign(h=hour_label)[["ORIGIN", "FL_DATE", "h", "temp_ORIGIN", "pres_ORIGIN"]]
         .drop_duplicates(["ORIGIN", "FL_DATE", "h"]).set_index(["ORIGIN", "FL_DATE", "h"]))
done = raw[raw.CANCELLED == 0]
for name, hours in {"scheduled arrival hour": done.CRS_ARR_TIME // 100,
                    "actual arrival hour": (done.ARR_TIME // 100) % 24}.items():
    m = o_key.reindex(pd.MultiIndex.from_arrays([done.DEST, done.FL_DATE, hours]))
    ok = m.temp_ORIGIN.notna().values
    agree = ((m.temp_ORIGIN.values == done.temp_DEST.values) & (m.pres_ORIGIN.values == done.pres_DEST.values))[ok].mean()
    print(f"Destination weather keyed on {name}: {agree:.1%} agreement ({ok.sum():,} comparable rows)")

Hour label of min / max mean temperature:


,coldest label hour,warmest label hour
ORIGIN,,
ATL,10,23
DFW,0,23
LAX,13,21


Destination weather keyed on scheduled arrival hour: 99.9% agreement (19,927 comparable rows)
Destination weather keyed on actual arrival hour: 69.7% agreement (19,797 comparable rows)


In [9]:
# True observation time of each snapshot = FL_DATE + scheduled local hour, read as UTC
df["origin_wx_obs_utc"] = (pd.to_datetime(df.FL_DATE) + pd.to_timedelta(df.CRS_DEP_TIME // 100, unit="h")).dt.tz_localize("UTC")
df["dest_wx_obs_utc"] = (pd.to_datetime(df.FL_DATE) + pd.to_timedelta(df.CRS_ARR_TIME // 100, unit="h")).dt.tz_localize("UTC")
df["origin_wx_age_h"] = ((df.sched_dep_utc - df.origin_wx_obs_utc).dt.total_seconds() / 3600).round(2)
df["dest_wx_age_h"] = ((df.sched_arr_utc - df.dest_wx_obs_utc).dt.total_seconds() / 3600).round(2)

assert (df.origin_wx_age_h > 0).all() and (df.dest_wx_age_h > 0).all(), "Some weather is from after the event"
age = pd.concat({
    "hub departures: origin wx age (h)": df.loc[df.ORIGIN.isin(HUBS)].groupby("ORIGIN").origin_wx_age_h.describe(),
    "hub arrivals: dest wx age (h)": df.loc[df.DEST.isin(HUBS)].groupby("DEST").dest_wx_age_h.describe(),
})
display(age.round(1))

count  mean  std  min  25%  50%  75%   max
                                  ORIGIN                                             
hub departures: origin wx age (h) ATL     8415.0   4.8  0.6  4.0  4.3  4.8  5.2   6.0
                                  DFW     7736.0   5.8  0.5  5.0  5.4  5.7  6.2   7.0
hub arrivals: dest wx age (h)     ATL     5429.0   6.0  5.3  4.0  4.4  4.8  5.3  30.0
                                  DFW     4032.0   6.5  4.0  5.0  5.4  5.8  6.3  30.9

### 6b. Hourly weather table rebuilt from the file

Each airport-hour appears many times across flights. Deduplicating gives a proper per-airport hourly series in UTC. It only covers hours that happened to have a sampled flight, so it has gaps. This lets downstream code use lagged weather from this file without trusting the flight-level join.

In [10]:
wx_cols = ["temp_c", "precip_mm", "wind_kmh", "wind_kt", "gust_kmh", "pres_hpa", "cond_code", "cond_group"]
stack = pd.concat([
    df[["ORIGIN", "origin_wx_obs_utc"] + [f"origin_wx_{c}" for c in wx_cols]]
      .set_axis(["airport", "obs_utc"] + wx_cols, axis=1),
    df[["DEST", "dest_wx_obs_utc"] + [f"dest_wx_{c}" for c in wx_cols]]
      .set_axis(["airport", "obs_utc"] + wx_cols, axis=1),
])
n_conflicts = stack.drop_duplicates().duplicated(["airport", "obs_utc"]).sum()
print(f"Airport-hours with conflicting values: {n_conflicts} (negligible; first value kept)")
wx_hourly_file = stack.drop_duplicates(["airport", "obs_utc"]).sort_values(["airport", "obs_utc"]).reset_index(drop=True)

hours_in_range = int((wx_hourly_file.obs_utc.max() - wx_hourly_file.obs_utc.min()).total_seconds() // 3600) + 1
cov = wx_hourly_file[wx_hourly_file.airport.isin(HUBS)].groupby("airport").size().to_frame("hours covered")
cov["share of all hours"] = (cov["hours covered"] / hours_in_range).round(3)
display(cov)

Airport-hours with conflicting values: 27 (negligible; first value kept)


,hours covered,share of all hours
airport,,
ATL,9411,0.416
DFW,8310,0.367


## 7. Add hub METARs from IEM (wind direction, visibility, ceiling)

The challenge requires **crosswind and tailwind relative to the runway**, which needs wind direction. The provided file has none. METARs from the Iowa Environmental Mesonet ASOS archive add:

* wind direction and speed in knots, plus gusts → runway headwind, tailwind and crosswind;
* visibility and the lowest cloud ceiling → FAA flight category (VFR/MVFR/IFR/LIFR), the main driver of reduced arrival rates;
* present-weather codes → thunderstorm (`TS`), freezing precipitation, snow.

The download is cached in `data/external/` and re-runs only if the file is missing. It takes no manual steps and works for any station code.

In [11]:
IEM_URL = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"
IEM_FIELDS = ["tmpc", "dwpc", "drct", "sknt", "gust", "vsby", "skyc1", "skyl1", "skyc2", "skyl2",
              "skyc3", "skyl3", "wxcodes", "p01m", "mslp"]


def fetch_iem_asos(station, start, end, cache_dir=EXT_DIR):
    # Download routine hourly METARs for one station (FAA 3-letter id) between two dates, UTC.
    path = cache_dir / f"iem_asos_{station}_{start}_{end}.csv"
    if not path.exists():
        s, e = pd.Timestamp(start), pd.Timestamp(end)
        params = [("station", station), *[("data", f) for f in IEM_FIELDS],
                  ("year1", s.year), ("month1", s.month), ("day1", s.day),
                  ("year2", e.year), ("month2", e.month), ("day2", e.day),
                  ("tz", "Etc/UTC"), ("format", "onlycomma"), ("latlon", "no"),
                  ("missing", "M"), ("trace", "T"), ("report_type", 3)]
        url = IEM_URL + "?" + urllib.parse.urlencode(params)
        with urllib.request.urlopen(url, timeout=300) as r:
            path.write_bytes(r.read())
    m = pd.read_csv(path, na_values=["M"], low_memory=False)
    m["valid"] = pd.to_datetime(m["valid"]).dt.tz_localize("UTC")
    return m


def ceiling_ft(m):
    # Lowest broken/overcast/vertical-visibility layer; NaN means no ceiling (unlimited).
    layers = []
    for i in (1, 2, 3):
        layers.append(m[f"skyl{i}"].where(m[f"skyc{i}"].isin(["BKN", "OVC", "VV"])))
    return pd.concat(layers, axis=1).min(axis=1)


def flight_category(ceil, vis):
    c = ceil.fillna(1e5)
    v = vis.fillna(10)
    return np.select(
        [(c < 500) | (v < 1), (c < 1000) | (v < 3), (c <= 3000) | (v <= 5)],
        ["LIFR", "IFR", "MVFR"], default="VFR")


def runway_wind_components(wind_dir_deg, speed_kt, runway_heading_deg):
    # Returns (headwind, crosswind). Headwind < 0 means tailwind. Wind direction is where it blows FROM.
    ang = np.deg2rad(wind_dir_deg - runway_heading_deg)
    return speed_kt * np.cos(ang), np.abs(speed_kt * np.sin(ang))


def build_hourly_metar(station, runway_heading_deg, start, end):
    m = fetch_iem_asos(station, start, end)
    m = m.rename(columns={"tmpc": "temp_c", "dwpc": "dewpt_c", "drct": "wind_dir_deg", "sknt": "wind_kt",
                          "gust": "gust_kt", "vsby": "vis_sm", "p01m": "precip_1h_mm", "mslp": "mslp_hpa"})
    m["precip_1h_mm"] = pd.to_numeric(m.precip_1h_mm.replace("T", 0.01))   # T = trace
    # Report at hh:52 counts for the next full hour (same convention as Meteostat)
    m["obs_utc"] = m.valid.dt.ceil("h")
    m = m.sort_values("valid").drop_duplicates("obs_utc", keep="last")

    m["ceiling_ft"] = ceiling_ft(m)
    m["flight_cat"] = flight_category(m.ceiling_ft, m.vis_sm)
    wx = m.wxcodes.fillna("")
    m["wx_thunder"] = wx.str.contains("TS").astype("int8")
    m["wx_freezing"] = wx.str.contains("FZ").astype("int8")
    m["wx_snow"] = wx.str.contains("SN").astype("int8")
    m["wx_rain"] = wx.str.contains("RA").astype("int8")

    calm = m.wind_kt.fillna(0) == 0
    variable = m.wind_dir_deg.isna() & ~calm   # VRB winds have no direction: assume the worst case
    head, cross = runway_wind_components(m.wind_dir_deg, m.wind_kt, runway_heading_deg)
    g_head, g_cross = runway_wind_components(m.wind_dir_deg, m.gust_kt.fillna(m.wind_kt), runway_heading_deg)
    m["runway_heading_deg"] = runway_heading_deg
    m["headwind_kt"] = np.where(calm, 0, np.where(variable, -m.wind_kt, head)).round(1)
    m["crosswind_kt"] = np.where(calm, 0, np.where(variable, m.wind_kt, cross)).round(1)
    m["tailwind_kt"] = (-m.headwind_kt).clip(lower=0) + 0.0   # + 0.0 turns -0.0 into 0.0
    m["gust_crosswind_kt"] = np.where(calm, 0, np.where(variable, m.gust_kt.fillna(m.wind_kt), g_cross)).round(1)
    m["wind_variable"] = variable.astype("int8")
    m.insert(0, "airport", station)
    keep = ["airport", "obs_utc", "valid", "temp_c", "dewpt_c", "wind_dir_deg", "wind_kt", "gust_kt",
            "runway_heading_deg", "headwind_kt", "tailwind_kt", "crosswind_kt", "gust_crosswind_kt",
            "wind_variable", "vis_sm", "ceiling_ft", "flight_cat", "wxcodes", "wx_thunder", "wx_freezing",
            "wx_snow", "wx_rain", "precip_1h_mm", "mslp_hpa"]
    return m[keep].reset_index(drop=True)


metar = None
if DOWNLOAD_METARS:
    start = pd.Timestamp(df.FL_DATE.min()) - pd.Timedelta(days=2)
    end = pd.Timestamp(df.FL_DATE.max()) + pd.Timedelta(days=2)
    try:
        metar = pd.concat([build_hourly_metar(ap, cfg["runway_heading_deg"], start.date(), end.date())
                           for ap, cfg in HUBS.items()], ignore_index=True)
        print(f"METAR hours: {len(metar):,}")
        display(metar.groupby("airport").agg(
            hours=("obs_utc", "size"), first=("obs_utc", "min"), last=("obs_utc", "max"),
            ifr_or_worse=("flight_cat", lambda s: s.isin(["IFR", "LIFR"]).mean()),
            thunder_hours=("wx_thunder", "sum"), crosswind_gt_15kt=("crosswind_kt", lambda s: (s > 15).mean()),
            tailwind_gt_10kt=("tailwind_kt", lambda s: (s > 10).mean())).round({"ifr_or_worse": 3, "crosswind_gt_15kt": 3, "tailwind_gt_10kt": 3}))
    except Exception as e:
        print(f"METAR download failed ({e}); continuing without it. Set DOWNLOAD_METARS=False to silence.")

METAR hours: 45,391


,hours,first,last,ifr_or_worse,thunder_hours,crosswind_gt_15kt,tailwind_gt_10kt
airport,,,,,,,
ATL,22692,2022-12-30 01:00:00+00:00,2025-08-02 00:00:00+00:00,0.080,289,0.008,0.042
DFW,22699,2022-12-30 01:00:00+00:00,2025-08-02 00:00:00+00:00,0.047,341,0.007,0.084


In [12]:
# Evidence 3: compare provided temperatures with METARs under both readings of the hour label
if metar is not None:
    file_hub = wx_hourly_file[wx_hourly_file.airport.isin(HUBS)][["airport", "obs_utc", "temp_c"]]
    as_utc = file_hub.merge(metar[["airport", "obs_utc", "temp_c"]], on=["airport", "obs_utc"], suffixes=("", "_metar"))
    # Alternative reading: the label hour was local time
    alt = file_hub.copy()
    alt["obs_utc"] = local_to_utc(alt.obs_utc.dt.tz_localize(None), alt.airport.map(TZ)).dt.floor("h")
    as_local = alt.merge(metar[["airport", "obs_utc", "temp_c"]], on=["airport", "obs_utc"], suffixes=("", "_metar"))
    for name, t in {"label hour is UTC": as_utc, "label hour is local": as_local}.items():
        ok = (t.temp_c - t.temp_c_metar).abs() <= 0.6
        print(f"{name:20s}: temperature within 0.6 °C in {ok.mean():.1%} of {len(t):,} hub-hours")

label hour is UTC   : temperature within 0.6 °C in 99.9% of 17,714 hub-hours
label hour is local : temperature within 0.6 °C in 15.6% of 17,719 hub-hours


## 8. Targets

* `dep_severe_45`: departure delay above 45 minutes. This is the brief's severe-delay class. Missing when the flight was cancelled before departing.
* `arr_severe_45`: same threshold on arrival delay.
* `disrupted_45`: **recommended main label**: severe departure delay **or** cancelled **or** diverted. From an operations point of view, a cancellation is the worst outcome. Dropping cancellations would hide exactly the weather days we care about.
* Delay-cause minutes: BTS only fills these when arrival delay is at least 15 minutes. For completed flights below that, the true value is 0, so we fill 0. Cancelled and diverted flights stay missing.

In [13]:
done = (df.CANCELLED == 0) & (df.DIVERTED == 0)
df["dep_severe_45"] = (df.DEP_DELAY > SEVERE_DELAY_MIN).astype("Int8").where(df.DEP_DELAY.notna())
df["arr_severe_45"] = (df.ARR_DELAY > SEVERE_DELAY_MIN).astype("Int8").where(df.ARR_DELAY.notna())
df["disrupted_45"] = ((df.DEP_DELAY > SEVERE_DELAY_MIN) | (df.CANCELLED == 1) | (df.DIVERTED == 1)).astype("int8")
df["cancelled_weather"] = (df.CANCELLATION_CODE == "B").astype("int8")

causes = ["CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"]
fill_zero = done & (df.ARR_DELAY < 15)
df.loc[fill_zero, causes] = df.loc[fill_zero, causes].fillna(0)

for c in ["CANCELLED", "DIVERTED"]:
    df[c] = df[c].astype("int8")

rates = df.assign(hub=np.where(df.ORIGIN.isin(HUBS), df.ORIGIN, "other origins")).groupby("hub")[
    ["dep_severe_45", "disrupted_45", "CANCELLED", "cancelled_weather"]].mean().round(4)
rates["flights"] = df.assign(hub=np.where(df.ORIGIN.isin(HUBS), df.ORIGIN, "other origins")).groupby("hub").size()
display(rates)

,dep_severe_45,disrupted_45,CANCELLED,cancelled_weather,flights
hub,,,,,
ATL,0.0935,0.1037,0.0088,0.0046,8415
DFW,0.1302,0.1472,0.0168,0.0146,7736
other origins,0.0964,0.1098,0.0128,0.0074,132122


These are **rates, not minutes**. The brief warns against comparing raw delay minutes between big and small airports. Every airport comparison downstream should use rates normalised by flight volume (share of flights disrupted, delay minutes per scheduled flight).

## 9. Hub scope and a leak-free join preview

`hub_role` tells you whether a row is a hub departure, a hub arrival, or both. Every flight also gets its runway heading at the hub end, so runway wind components can be attached for any forecast horizon.

The preview below shows the **as-of join** the feature-building step should use. For each hub departure and forecast horizon `k`, it takes the latest METAR observed **at or before** `scheduled time − k hours`. The assert proves no feature comes from after the moment the forecast would be issued.

In [14]:
df["hub_role"] = np.select(
    [df.ORIGIN.isin(HUBS) & df.DEST.isin(HUBS), df.ORIGIN.isin(HUBS), df.DEST.isin(HUBS)],
    ["both", "hub_departure", "hub_arrival"], default="non_hub")
df["hub_role"] = df.hub_role.astype("category")
heading = {k: v["runway_heading_deg"] for k, v in HUBS.items()}
df["origin_runway_heading_deg"] = df.ORIGIN.map(heading)
df["dest_runway_heading_deg"] = df.DEST.map(heading)
display(df.hub_role.value_counts().to_frame("rows"))


def asof_weather(events, time_col, airport_col, wx, horizon_h, cols):
    # Attach the latest observation available `horizon_h` hours before each event time.
    ev = events[[time_col, airport_col]].copy()
    ev["issue_utc"] = (ev[time_col] - pd.Timedelta(hours=horizon_h)).astype("datetime64[ns, UTC]")
    ev = ev.reset_index().sort_values("issue_utc")
    w = wx[["airport", "valid"] + cols].sort_values("valid")
    w["valid"] = w["valid"].astype("datetime64[ns, UTC]")
    j = pd.merge_asof(ev, w, left_on="issue_utc", right_on="valid", left_by=airport_col, right_by="airport",
                      direction="backward", tolerance=pd.Timedelta(hours=3))
    return j.set_index("index").sort_index()


if metar is not None:
    hub_dep = df[df.ORIGIN.isin(HUBS)]
    preview = asof_weather(hub_dep, "sched_dep_utc", "ORIGIN", metar, horizon_h=3,
                           cols=["flight_cat", "crosswind_kt", "tailwind_kt", "wx_thunder"])
    assert (preview.valid.dropna() <= preview.issue_utc[preview.valid.notna()]).all()
    print(f"3-hour-ahead METAR found for {preview.valid.notna().mean():.1%} of hub departures")
    display(preview.head())

,rows
hub_role,
non_hub,122974
hub_departure,15838
hub_arrival,9148
both,313


3-hour-ahead METAR found for 100.0% of hub departures


,sched_dep_utc,ORIGIN,issue_utc,airport,valid,flight_cat,crosswind_kt,tailwind_kt,wx_thunder
index,,,,,,,,,
0,2023-01-16 20:50:00+00:00,ATL,2023-01-16 17:50:00+00:00,ATL,2023-01-16 16:52:00+00:00,VFR,6.6,2.4,0.0
1,2023-01-30 12:26:00+00:00,ATL,2023-01-30 09:26:00+00:00,ATL,2023-01-30 08:52:00+00:00,LIFR,0.0,0.0,0.0
2,2023-01-30 21:00:00+00:00,ATL,2023-01-30 18:00:00+00:00,ATL,2023-01-30 17:52:00+00:00,MVFR,0.0,0.0,0.0
16,2023-01-31 20:10:00+00:00,ATL,2023-01-31 17:10:00+00:00,ATL,2023-01-31 16:52:00+00:00,MVFR,2.7,7.5,0.0
17,2023-01-10 21:40:00+00:00,ATL,2023-01-10 18:40:00+00:00,ATL,2023-01-10 17:52:00+00:00,VFR,1.4,7.9,0.0


## 10. Export

In [15]:
COLUMN_ORDER = (
    ["FL_DATE", "OP_UNIQUE_CARRIER", "OP_CARRIER_FL_NUM", "TAIL_NUM", "ORIGIN", "DEST", "origin_icao", "dest_icao",
     "hub_role", "origin_tz", "dest_tz", "origin_runway_heading_deg", "dest_runway_heading_deg",
     "CRS_DEP_TIME", "CRS_ARR_TIME", "CRS_ELAPSED_TIME", "DISTANCE",
     "sched_dep_utc", "sched_arr_utc", "sched_dep_local", "sched_arr_local",
     "sched_dep_hour_local", "sched_arr_hour_local", "month", "day_of_week",
     "dep_severe_45", "arr_severe_45", "disrupted_45", "cancelled_weather",
     "DEP_TIME", "DEP_DELAY", "TAXI_OUT", "WHEELS_OFF", "WHEELS_ON", "TAXI_IN", "ARR_TIME", "ARR_DELAY",
     "actual_dep_utc", "actual_arr_utc", "ACTUAL_ELAPSED_TIME", "AIR_TIME",
     "CANCELLED", "CANCELLATION_CODE", "DIVERTED"] + causes
    + [c for c in df.columns if c.startswith("origin_wx_")] + [c for c in df.columns if c.startswith("dest_wx_")]
)
assert set(COLUMN_ORDER) == set(df.columns), set(df.columns) ^ set(COLUMN_ORDER)
clean = df[COLUMN_ORDER].copy()
for c in ["OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "origin_icao", "dest_icao", "origin_tz", "dest_tz", "CANCELLATION_CODE"]:
    clean[c] = clean[c].astype("category")
clean["OP_CARRIER_FL_NUM"] = clean.OP_CARRIER_FL_NUM.astype("int32")

key = ["FL_DATE", "OP_UNIQUE_CARRIER", "OP_CARRIER_FL_NUM", "ORIGIN", "DEST", "CRS_DEP_TIME"]
assert not clean.duplicated(key).any(), "Duplicate flights"

hubs = clean[clean.hub_role != "non_hub"].reset_index(drop=True)

outputs = {
    "flights_clean_all.parquet": clean,
    "flights_clean_hubs.parquet": hubs,
    "weather_hourly_from_dataset.parquet": wx_hourly_file,
}
if metar is not None:
    outputs["hub_metar_hourly.parquet"] = metar
for name, frame in outputs.items():
    frame.to_parquet(OUT_DIR / name, index=False)
hubs.to_csv(OUT_DIR / "flights_clean_hubs.csv", index=False)   # for spreadsheet users

for p in sorted(OUT_DIR.iterdir()):
    print(f"{p.name:40s} {p.stat().st_size / 1e6:6.1f} MB")

cleaning_report.json                        0.0 MB
data_dictionary.csv                         0.0 MB
flights_clean_all.parquet                  15.1 MB
flights_clean_hubs.csv                     12.4 MB
flights_clean_hubs.parquet                  2.9 MB
hub_metar_hourly.parquet                    1.1 MB
weather_hourly_from_dataset.parquet         1.9 MB


In [16]:
# Data dictionary: one row per exported flight column, with its role so the modelling step knows what is a safe input
DERIVED = {
    "origin_icao": ("id", "ICAO code from airportsdata"), "dest_icao": ("id", "ICAO code from airportsdata"),
    "hub_role": ("id", "hub_departure / hub_arrival / both / non_hub"),
    "origin_tz": ("schedule", "IANA time zone"), "dest_tz": ("schedule", "IANA time zone"),
    "origin_runway_heading_deg": ("schedule", "Assumed runway heading (brief); NaN for non-hubs"),
    "dest_runway_heading_deg": ("schedule", "Assumed runway heading (brief); NaN for non-hubs"),
    "sched_dep_utc": ("schedule", "Scheduled departure, UTC"), "sched_arr_utc": ("schedule", "Scheduled departure + CRS_ELAPSED_TIME, UTC"),
    "sched_dep_local": ("schedule", "Scheduled departure, origin local"), "sched_arr_local": ("schedule", "Scheduled arrival, destination local"),
    "sched_dep_hour_local": ("schedule", "0-23"), "sched_arr_hour_local": ("schedule", "0-23"),
    "month": ("schedule", "1-12"), "day_of_week": ("schedule", "0 = Monday"),
    "dep_severe_45": ("target", f"DEP_DELAY > {SEVERE_DELAY_MIN} min; NA if not departed"),
    "arr_severe_45": ("target", f"ARR_DELAY > {SEVERE_DELAY_MIN} min; NA if not arrived"),
    "disrupted_45": ("target", "Severe dep delay OR cancelled OR diverted (recommended main label)"),
    "cancelled_weather": ("target", "Cancelled with code B (weather)"),
    "actual_dep_utc": ("outcome", "sched_dep_utc + DEP_DELAY"), "actual_arr_utc": ("outcome", "sched_arr_utc + ARR_DELAY"),
}
WX_DESC = {"temp_c": "Air temperature, °C", "precip_mm": "Precipitation in the hour, mm", "wind_kmh": "Mean wind speed, km/h",
           "wind_kt": "Mean wind speed, knots", "gust_kmh": "Peak gust, km/h (NaN = not reported)",
           "pres_hpa": "Sea-level pressure, hPa", "cond_code": "Meteostat condition code 1-27",
           "cond_group": "none / rain / winter / low_visibility / convective",
           "obs_utc": "TRUE observation time of this snapshot (see section 6)",
           "age_h": "Hours between observation and the scheduled event; valid for horizon k only if age >= k"}
rows = []
for c in clean.columns:
    if c in KEEP:
        role, desc = KEEP[c]
    elif c in DERIVED:
        role, desc = DERIVED[c]
    else:
        side, var = c.split("_wx_")
        role, desc = "weather", f"{side.capitalize()} airport (Meteostat via provided file): {WX_DESC[var]}"
    rows.append({"column": c, "role": role, "dtype": str(clean[c].dtype),
                 "null_frac": round(clean[c].isna().mean(), 4), "description": desc})
dictionary = pd.DataFrame(rows)
dictionary.to_csv(OUT_DIR / "data_dictionary.csv", index=False)
display(dictionary.role.value_counts().to_frame("columns"))

report = {
    "raw_rows": len(raw), "raw_columns": raw.shape[1],
    "clean_rows_all": len(clean), "clean_rows_hubs": len(hubs), "clean_columns": clean.shape[1],
    "dropped_columns": dropped,
    "hubs": HUBS, "severe_delay_threshold_min": SEVERE_DELAY_MIN,
    "weather_timing_finding": "Provided weather is Meteostat UTC-hour data joined on local scheduled hour; "
                              "true obs time stored in *_wx_obs_utc, age in *_wx_age_h.",
    "sentinels_replaced": ["wpgt 0 -> NaN", "coco 0 -> NaN", "snow dropped (all 0)"],
    "metar_hours": None if metar is None else int(len(metar)),
}
(OUT_DIR / "cleaning_report.json").write_text(json.dumps(report, indent=2, default=str))
print("Wrote data_dictionary.csv and cleaning_report.json")

,columns
role,
schedule,20
outcome,20
weather,20
id,5
target,4


Wrote data_dictionary.csv and cleaning_report.json


In [17]:
# Round-trip check: exported files load back with the same shape and timezone-aware timestamps
back = pd.read_parquet(OUT_DIR / "flights_clean_hubs.parquet")
assert back.shape == hubs.shape
assert str(back.sched_dep_utc.dt.tz) == "UTC"
print("Exports verified.", back.shape)

Exports verified. (25299, 69)


## 11. Known limitations

| Limitation | Effect | Fix |
|---|---|---|
| Flights are a sample of about 1% | Hourly arrival counts can't be measured, so no AAR label from this file | Download full BTS monthly files for ATL/DFW (`PLAN.md`, step 1) |
| Provided weather is 4–30 h stale (time zone mislabel) | Not "current" weather; its lag varies by row | Use `hub_metar_hourly.parquet` and TAFs with an as-of join |
| No wind direction in provided weather | No runway components from it | Taken from METARs (section 7) |
| Single assumed runway heading per hub | Real airports turn traffic around when tailwind gets high; we follow the brief | Optionally evaluate both runway ends or use OurAirports runway data |
| `NAS_DELAY` mixes weather with volume | Weather-attributable delay is underestimated if only `WEATHER_DELAY` is used | Report both; NOAA Storm Events gives an independent weather label |
| Gust is missing for most rows | Gust features from the file are unreliable | Use METAR gusts |

Next steps are in **`PLAN.md`**.